In [1]:
import json

with open('adj_demo.json', 'r') as f:
    adj = json.load(f)

In [2]:
# libs
import json
from bisect import bisect_left
from collections import deque
from math import inf
from typing import List, Tuple, Dict, Optional, Any, Hashable

## calculate_earlist_arrival() Core Time Window Priority Search Implementation

In [6]:
Node = Hashable
eID = Any

tempAdjItem = Tuple[int, Node, eID]
tempAdj = Dict[Node, List[tempAdjItem]]

prevRecord = Dict[str, Any]

In [7]:
# optional, depends on the data format
with open('adj_demo.json', 'r') as f:
    adj = json.load(f)

In [8]:
def first_edge_idx(out_edges: List[tempAdjItem], current_t: int) -> int:  # t:
    edge_times = [item[0] for item in out_edges] # get unix val of the node
    return bisect_left(edge_times, current_t) # return the index of the closest edge

In [9]:
def calculate_earlist_arrival(
    adj: tempAdj,
    source: Node,
    start_time: int,
    target: Optional[Node] = None
) -> Tuple[Dict[Node, int], Dict[Node, Optional[prevRecord]]]:
    arrival_t = {source: start_time}
    prev = {source: None}

    queue = deque([source])
    in_queue = {source}

    while queue:
        cur_node = queue.popleft()
        in_queue.discard(cur_node)

        cur_time = arrival_t[cur_node]

        out_edges = adj.get(cur_node, [])
        if not out_edges:
            continue

        start_idx = first_edge_idx(out_edges, cur_time)
        for edge_t, next_node, eid in out_edges[start_idx:]:
            if edge_t < arrival_t.get(next_node, inf):
                arrival_t[next_node] = edge_t
                prev[next_node] = {
                    "prev_node": cur_node,
                    "prev_edge": eid,
                    "depart_t": edge_t,
                    "arrival_t": edge_t,
                }

                if next_node not in in_queue:
                    queue.append(next_node)
                    in_queue.add(next_node)

    return arrival_t, prev

## extract_path_sequence() Path Backtracking and Reconstruction

In [10]:
def extract_path_sequence(
    source: Node,
    target: Node,
    prev: Dict[Node, Optional[prevRecord]],
    arrival_times: Optional[Dict[Node, int]] = None
) -> Dict[str, Any]:
    if source == target:
        return {
            "found": True,
            "path_node": [source],
            "path_edge": [],
            "hop_count": 0,
            "final_arrival_t": arrival_times.get(target) if arrival_times else None,
        }

    if target not in prev:
        return {
            "found": False,
            "path_node": [],
            "path_edge": [],
            "hop_count": 0,
            "final_arrival_t": None,
        }

    reversed_nodes = [target]
    reversed_edges = []
    pivot = target

    while pivot != source:
        record = prev.get(pivot)
        if record is None:
            return {
                "found": False,
                "path_node": [],
                "path_edge": [],
                "hop_count": 0,
                "final_arrival_t": None,
            }

        prev_node = record["prev_node"]
        reversed_edges.append({
            "src": prev_node,
            "tgt": pivot,
            "edge_id": record["prev_edge"],
            "depart_t": record["depart_t"],
            "arrival_t": record["arrival_t"],
        })
        reversed_nodes.append(prev_node)
        pivot = prev_node

    path_nodes = list(reversed(reversed_nodes))
    path_edges = list(reversed(reversed_edges))
    hop_count = len(path_nodes) - 1

    return {
        "found": True,
        "path_node": path_nodes,
        "path_edge": path_edges,
        "hop_count": hop_count,
        "final_arrival_t": arrival_times.get(target) if arrival_times else None,
    }

## Test

In [11]:
adj_time = {
    "1": [(1254192988, '4', 'e0000001'),   # cols: time, target, edge_id
          (1254193612, '2', 'e0000003')
          ],
    "2":  [(1254194656, '4', 'e0000002'),
          (1254196156, '1', 'e0000004'),
    ]
}

In [12]:
adj_test = {
    "A" : [(2, "B", "e1"), (5, "C", "e2")],
    "B": [(4, "C", "e3")],
    "C": [(7, "D", "e5")],
    "D":[]

}

In [13]:
from re import S
arrival_t, prev = calculate_earlist_arrival(
    adj = adj_time,
    source = '1',
    start_time=0,
    target='4',
)

path_info = extract_path_sequence(
    source = '1',
    target='4',
    prev=prev,
    arrival_times=arrival_t,
)

print(arrival_t)
print(prev)
print(path_info)

{'1': 0, '4': 1254192988, '2': 1254193612}
{'1': None, '4': {'prev_node': '1', 'prev_edge': 'e0000001', 'depart_t': 1254192988, 'arrival_t': 1254192988}, '2': {'prev_node': '1', 'prev_edge': 'e0000003', 'depart_t': 1254193612, 'arrival_t': 1254193612}}
{'found': True, 'path_node': ['1', '4'], 'path_edge': [{'src': '1', 'tgt': '4', 'edge_id': 'e0000001', 'depart_t': 1254192988, 'arrival_t': 1254192988}], 'hop_count': 1, 'final_arrival_t': 1254192988}


In [14]:
arrival_t, prev = calculate_earlist_arrival(
    adj = adj_test,
    source = 'A',
    start_time=0,
    target='D',
)

path_info = extract_path_sequence(
    source = 'A',
    target='D',
    prev=prev,
    arrival_times=arrival_t,
)

print(arrival_t)
print(prev)
print(path_info)

{'A': 0, 'B': 2, 'C': 4, 'D': 7}
{'A': None, 'B': {'prev_node': 'A', 'prev_edge': 'e1', 'depart_t': 2, 'arrival_t': 2}, 'C': {'prev_node': 'B', 'prev_edge': 'e3', 'depart_t': 4, 'arrival_t': 4}, 'D': {'prev_node': 'C', 'prev_edge': 'e5', 'depart_t': 7, 'arrival_t': 7}}
{'found': True, 'path_node': ['A', 'B', 'C', 'D'], 'path_edge': [{'src': 'A', 'tgt': 'B', 'edge_id': 'e1', 'depart_t': 2, 'arrival_t': 2}, {'src': 'B', 'tgt': 'C', 'edge_id': 'e3', 'depart_t': 4, 'arrival_t': 4}, {'src': 'C', 'tgt': 'D', 'edge_id': 'e5', 'depart_t': 7, 'arrival_t': 7}], 'hop_count': 3, 'final_arrival_t': 7}


In [15]:
arrival, prev = calculate_earlist_arrival(adj, source='1', start_time=1254192988, target='100')
print(arrival)

{'1': 1254192988, '4': 1254192988, '2': 1254202612, '25': 1254259818, '16': 1254271421, '22': 1254273152, '3': 1254274939, '28': 1254279260, '27': 1254302991, '7': 1254392595, '32': 1254395565, '37': 1254441471, '40': 1254480209, '42': 1254481887, '21': 1254560006, '44': 1254606584, '45': 1254529848, '71': 1254880722, '85': 1254896616, '78': 1254836203, '100': 1255093192, '102': 1255095047, '94': 1255096320, '66': 1254737060, '83': 1254878745, '1004': 1255168840, '126': 1255214907, '121': 1255232089, '450': 1255292924, '158': 1255296773, '266': 1255388219, '184': 1255310665, '132': 1255484785, '336': 1255584057, '347': 1255585369, '11': 1255399048, '344': 1255604261, '370': 1255609086, '332': 1255652953, '416': 1255669488, '462': 1255739301, '143': 1255569554, '438': 1255776235, '65': 1254743888, '540': 1255826740, '307': 1255857166, '625': 1255865151, '619': 1255900219, '709': 1255949888, '519': 1255956991, '350': 1255659885, '728': 1255983952, '454': 1255754771, '290': 1255507913, '6